# Empirical Analysis: Employment Prevalence Under Age Distribution Shift

This notebook reproduces the ACS analysis from the paper. We train a logistic regression
on ACS employment data, calibrate it with Isotonic Regression and MCGrad, then measure
prevalence estimation bias under synthetic age distribution shift.

In [ ]:
import os

import numpy as np
import pandas as pd
from sklearn import metrics as skmetrics
from sklearn.model_selection import train_test_split
from mcgrad import metrics, methods, plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

from helpers import (
    BINARY_COLUMNS,
    CATEGORICAL_COLUMNS,
    LABEL_COLUMN,
    NUMERICAL_COLUMNS,
    configure_logging,
    create_logistic_pipeline,
    load_acs_employment_data,
    setup_plotting,
    compute_rogan_gladen_estimate,
    estimate_classifier_error_rates,
    calibrate_threshold_prevalence_matching,
)

os.makedirs('../paper/images', exist_ok=True)
setup_plotting()
configure_logging()

## 1. Data Loading

In [ ]:
# Load data from multiple states and years for more robust analysis
# Training states: Large, geographically diverse states
# OOD states: Held out for out-of-distribution evaluation

TRAIN_STATES = ["TX", "MI", "PA", "OH", "IL", "GA", "NC", "VA"]
OOD_STATES = ["CA", "NY", "FL", "WA", "AZ", "CO"]

# Use multiple survey years for more data
SURVEY_YEARS = ["2016", "2017", "2018"]

print(f"Loading training data: {len(TRAIN_STATES)} states x {len(SURVEY_YEARS)} years...")
train_source_df = load_acs_employment_data(
    states=TRAIN_STATES, 
    survey_years=SURVEY_YEARS,
    include_state=True,
    include_year=True,
)
print(f"Training states: {len(train_source_df)} total samples")
print(f"  By state: {train_source_df['STATE'].value_counts().to_dict()}")
print(f"  By year: {train_source_df['YEAR'].value_counts().to_dict()}")

print(f"\nLoading OOD data: {len(OOD_STATES)} states x {len(SURVEY_YEARS)} years...")
ood_df = load_acs_employment_data(
    states=OOD_STATES, 
    survey_years=SURVEY_YEARS,
    include_state=True,
    include_year=True,
)
print(f"OOD states: {len(ood_df)} total samples")
print(f"  By state: {ood_df['STATE'].value_counts().to_dict()}")

## 2. Model Training and Calibration

In [ ]:
# Split training data into train/calibration and held-out test sets
from sklearn.model_selection import train_test_split

# Stratify by both label and state to ensure balanced representation
train_df, test_df = train_test_split(
    train_source_df, 
    test_size=0.30, 
    random_state=42, 
    stratify=train_source_df[[LABEL_COLUMN, "STATE"]].apply(tuple, axis=1)
)

# Further split train into model training and calibration sets
model_train_df, calibration_df = train_test_split(
    train_df, 
    test_size=0.30, 
    random_state=42, 
    stratify=train_df[[LABEL_COLUMN, "STATE"]].apply(tuple, axis=1)
)

print(f"Model training set: {len(model_train_df):,} samples")
print(f"Calibration set: {len(calibration_df):,} samples")
print(f"In-distribution test set: {len(test_df):,} samples")
print(f"\nState distribution in test set:")
print(test_df["STATE"].value_counts().to_dict())

In [ ]:
# Train the base logistic regression model
# Exclude STATE and YEAR from features (they're for segmentation analysis, not prediction)
feature_cols = [c for c in model_train_df.columns if c not in [LABEL_COLUMN, "STATE", "YEAR"]]

X_train = model_train_df[feature_cols]
y_train = model_train_df[LABEL_COLUMN].astype(int).to_numpy()

logistic_pipeline = create_logistic_pipeline()
logistic_pipeline.fit(X_train, y_train)
print(f"Base model trained on {len(feature_cols)} features")

In [ ]:
# Generate base model predictions for all datasets
BASE_MODEL_COL = 'base_model_prediction'

for df in [calibration_df, test_df, ood_df]:
    # Use only the feature columns (exclude STATE, YEAR, and label)
    X = df[[c for c in df.columns if c not in [LABEL_COLUMN, "STATE", "YEAR"]]]
    df[BASE_MODEL_COL] = logistic_pipeline.predict_proba(X)[:, 1]

print("Base model predictions generated for all datasets")

In [ ]:
# Fit calibration methods on the calibration set

# 1. Isotonic Regression (global calibration)
isotonic_regression = methods.IsotonicRegression().fit(
    calibration_df,
    BASE_MODEL_COL,
    LABEL_COLUMN,
)
print("Isotonic regression fitted")

# 2. MCGrad (multicalibration)
categorical_segment_features = CATEGORICAL_COLUMNS + BINARY_COLUMNS
numerical_segment_features = NUMERICAL_COLUMNS

mcgrad = methods.MCGrad()
mcgrad = mcgrad.fit(
    calibration_df,
    BASE_MODEL_COL,
    LABEL_COLUMN,
    categorical_feature_column_names=categorical_segment_features,
    numerical_feature_column_names=numerical_segment_features,
)
print("MCGrad fitted")

In [ ]:
# Generate calibrated predictions for all evaluation datasets
IR_COL = 'isotonic_prediction'
MCGRAD_COL = 'mcgrad_prediction'

for df in [test_df, ood_df]:
    # Isotonic regression predictions
    df[IR_COL] = isotonic_regression.predict(df, BASE_MODEL_COL)
    
    # MCGrad predictions
    df[MCGRAD_COL] = mcgrad.predict(
        df=df,
        prediction_column_name=BASE_MODEL_COL,
        categorical_feature_column_names=categorical_segment_features,
        numerical_feature_column_names=numerical_segment_features,
    )

print("Calibrated predictions generated")

In [ ]:
# Calibrate threshold and estimate TPR/FPR from calibration set

# Use prevalence-matching threshold: ensures Classify & Count is unbiased on calibration distribution
THRESHOLD = calibrate_threshold_prevalence_matching(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[BASE_MODEL_COL],
)
print(f"Calibrated threshold (prevalence-matching): {THRESHOLD:.4f}")
print(f"  (vs naive 0.5 threshold)")

# Estimate TPR and FPR at calibrated threshold for Rogan-Gladen
calibration_tpr, calibration_fpr = estimate_classifier_error_rates(
    labels=calibration_df[LABEL_COLUMN],
    predictions=calibration_df[BASE_MODEL_COL],
    threshold=THRESHOLD,
)

print(f"\nCalibration set error rates at threshold={THRESHOLD:.4f}:")
print(f"  TPR (sensitivity): {calibration_tpr:.4f}")
print(f"  FPR (1-specificity): {calibration_fpr:.4f}")

# Verify: at this threshold, apparent prevalence should match true prevalence
apparent_prev = (calibration_df[BASE_MODEL_COL] >= THRESHOLD).mean()
true_prev = calibration_df[LABEL_COLUMN].mean()
print(f"\nVerification:")
print(f"  True prevalence in calibration set: {true_prev:.4f}")
print(f"  Apparent prevalence at calibrated threshold: {apparent_prev:.4f}")

## 3. Prevalence Estimation Under Synthetic Age Distribution Shift

We create synthetic target populations by resampling with different age distributions,
then compare prevalence estimation bias across methods.

In [ ]:
def compute_all_prevalence_estimates(
    target_df: pd.DataFrame,
    base_col: str,
    ir_col: str,
    mcgrad_col: str,
    label_col: str,
    cal_tpr: float,
    cal_fpr: float,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Compute prevalence estimates using all methods."""
    true_prevalence = target_df[label_col].mean()
    
    # Raw base model scores
    raw_estimate = target_df[base_col].mean()
    
    # Classify and count (threshold-based, no adjustment)
    binary_preds = (target_df[base_col] >= threshold).astype(int)
    classify_count = binary_preds.mean()
    
    # Rogan-Gladen adjusted count
    rogan_gladen = compute_rogan_gladen_estimate(classify_count, cal_tpr, cal_fpr)
    
    # Isotonic regression calibrated scores
    isotonic_estimate = target_df[ir_col].mean()
    
    # MCGrad calibrated scores
    mcgrad_estimate = target_df[mcgrad_col].mean()
    
    return {
        "True Prevalence": true_prevalence,
        "Raw Scores": raw_estimate,
        "Classify & Count": classify_count,
        "Rogan-Gladen": rogan_gladen,
        "Isotonic Regression": isotonic_estimate,
        "MCGrad": mcgrad_estimate,
    }


def compute_bias_table(estimates: dict[str, float]) -> pd.DataFrame:
    """Create a DataFrame showing estimates and bias for each method."""
    true_prev = estimates["True Prevalence"]
    rows = []
    for method, estimate in estimates.items():
        if method == "True Prevalence":
            continue
        bias = estimate - true_prev
        rows.append({
            "Method": method,
            "Estimate": estimate,
            "Bias": bias,
            "Relative Bias (%)": 100 * bias / true_prev if true_prev > 0 else 0,
        })
    return pd.DataFrame(rows).set_index("Method")


# Methods to compare
methods_to_plot = ["Raw Scores", "Classify & Count", "Rogan-Gladen", "Isotonic Regression", "MCGrad"]

colors_map = {
    "Raw Scores": "#377eb8",
    "Classify & Count": "#ff7f00",
    "Rogan-Gladen": "#984ea3",
    "Isotonic Regression": "#e41a1c",
    "MCGrad": "#4daf4a",
}

In [ ]:
# Create synthetic populations by resampling with different age distributions
# We use the test_df which already has all prediction columns

def resample_with_age_shift(
    df: pd.DataFrame,
    age_col: str = "AGEP",
    shift: str = "original",
    n_samples: int = 20_000,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Resample df with importance weights that shift the age distribution.
    
    shift options:
      - "original": uniform weights (baseline)
      - "young": heavily oversample ages 16-30
      - "old": heavily oversample ages 60+
      - "bimodal": oversample both young and old, undersample middle
    """
    ages = df[age_col].values
    
    if shift == "original":
        weights = np.ones(len(df))
    elif shift == "young":
        # Exponentially favor younger ages
        weights = np.exp(-0.08 * (ages - 16))
        weights = np.where(ages <= 30, weights * 5, weights)
    elif shift == "old":
        # Exponentially favor older ages
        weights = np.exp(0.08 * (ages - 50))
        weights = np.where(ages >= 60, weights * 5, weights)
    elif shift == "bimodal":
        # Favor both tails — young and old
        center = 40
        weights = np.exp(0.04 * np.abs(ages - center))
        weights = np.where((ages <= 25) | (ages >= 65), weights * 3, weights)
    else:
        raise ValueError(f"Unknown shift: {shift}")
    
    weights = weights / weights.sum()
    
    return df.sample(n=n_samples, weights=weights, replace=True, random_state=random_state)


# Define the synthetic scenarios
age_shifts = {
    "Original": "original",
    "Young-skewed": "young",
    "Old-skewed": "old",
    "Bimodal (young+old)": "bimodal",
}

# Generate synthetic populations from the IN-DISTRIBUTION test set
synthetic_populations = {}
for label, shift in age_shifts.items():
    synthetic_populations[label] = resample_with_age_shift(test_df, shift=shift)

# Show the age distributions we created
fig = go.Figure()
for label, syn_df in synthetic_populations.items():
    age_counts = syn_df['AGEP'].value_counts().sort_index()
    fig.add_trace(go.Scatter(
        x=age_counts.index, y=age_counts.values,
        mode='lines', name=label,
    ))

fig.update_layout(
    title="Age Distributions of Synthetic Populations",
    xaxis_title="Age (AGEP)",
    yaxis_title="Count",
    height=400, width=900,
)
fig.show()

# Print true employment rates
for label, syn_df in synthetic_populations.items():
    print(f"{label}: n={len(syn_df):,}, true employment rate = {syn_df[LABEL_COLUMN].mean():.1%}, "
          f"mean age = {syn_df['AGEP'].mean():.1f}")

In [ ]:
# Compute prevalence estimates for each synthetic population
synthetic_results = {}

for scenario_label, syn_df in synthetic_populations.items():
    estimates = compute_all_prevalence_estimates(
        target_df=syn_df,
        base_col=BASE_MODEL_COL,
        ir_col=IR_COL,
        mcgrad_col=MCGRAD_COL,
        label_col=LABEL_COLUMN,
        cal_tpr=calibration_tpr,
        cal_fpr=calibration_fpr,
        threshold=THRESHOLD,
    )
    synthetic_results[scenario_label] = estimates
    print(f"\n{scenario_label}:")
    print(f"  True prevalence: {estimates['True Prevalence']:.4f}")
    display(compute_bias_table(estimates).round(4))

In [ ]:
# Visualization: Prevalence estimates vs true prevalence under age distribution shift
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Prevalence Estimates", "Estimation Bias"),
    horizontal_spacing=0.12,
)

scenarios = list(synthetic_results.keys())
# Left panel: estimates vs true prevalence
true_prevs = [synthetic_results[s]["True Prevalence"] for s in scenarios]
fig.add_trace(
    go.Scatter(x=scenarios, y=true_prevs, mode='lines+markers',
               name='True Prevalence', line=dict(color='black', width=3, dash='dash'),
               marker=dict(size=10, symbol='diamond')),
    row=1, col=1,
)

for method in methods_to_plot:
    estimates = [synthetic_results[s][method] for s in scenarios]
    fig.add_trace(
        go.Scatter(x=scenarios, y=estimates, mode='lines+markers',
                   name=method, marker=dict(size=8),
                   line=dict(color=colors_map[method])),
        row=1, col=1,
    )

# Right panel: bias (percentage points)
for method in methods_to_plot:
    biases = [(synthetic_results[s][method] - synthetic_results[s]["True Prevalence"]) * 100
              for s in scenarios]
    fig.add_trace(
        go.Bar(x=scenarios, y=biases, name=method, showlegend=False,
               marker_color=colors_map[method],
               text=[f"{b:+.2f}" for b in biases], textposition='outside'),
        row=1, col=2,
    )

fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

fig.update_yaxes(title_text="Employment Rate", tickformat=".1%", row=1, col=1)
fig.update_yaxes(title_text="Bias (percentage points)", row=1, col=2)
fig.update_layout(
    title="Prevalence Estimation Under Synthetic Age Distribution Shift (In-Distribution Data)",
    height=550, width=1200,
    barmode='group',
)
fig.show()

In [ ]:
# Repeat on OOD data: synthetic age shifts applied to held-out states
# This combines distribution shift on TWO axes: geography (OOD states) AND age

synthetic_populations_ood = {}
for label, shift in age_shifts.items():
    synthetic_populations_ood[label] = resample_with_age_shift(ood_df, shift=shift)

synthetic_results_ood = {}
for scenario_label, syn_df in synthetic_populations_ood.items():
    estimates = compute_all_prevalence_estimates(
        target_df=syn_df,
        base_col=BASE_MODEL_COL,
        ir_col=IR_COL,
        mcgrad_col=MCGRAD_COL,
        label_col=LABEL_COLUMN,
        cal_tpr=calibration_tpr,
        cal_fpr=calibration_fpr,
        threshold=THRESHOLD,
    )
    synthetic_results_ood[scenario_label] = estimates

# Side-by-side: In-dist vs OOD under age shift
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("In-Distribution (age-shifted)", "OOD States (age-shifted)"),
    shared_yaxes=True,
)

for col, results in enumerate([synthetic_results, synthetic_results_ood], start=1):
    for i, method in enumerate(methods_to_plot):
        biases = [(results[s][method] - results[s]["True Prevalence"]) * 100
                  for s in scenarios]
        fig.add_trace(
            go.Bar(x=scenarios, y=biases, name=method,
                   marker_color=colors_map[method],
                   showlegend=(col == 1)),
            row=1, col=col,
        )

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_yaxes(title_text="Bias (percentage points)", row=1, col=1)
fig.update_layout(
    title="Prevalence Estimation Bias Under Age Distribution Shift: In-Dist vs OOD",
    height=500, width=1200,
    barmode='group',
)
fig.show()

# Save as static image with matplotlib
import matplotlib.pyplot as plt
import numpy as np

fig_mpl, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

x = np.arange(len(scenarios))
n_methods = len(methods_to_plot)
width = 0.8 / n_methods

for panel_idx, (ax, results, title) in enumerate([
    (ax1, synthetic_results, "In-Distribution (age-shifted)"),
    (ax2, synthetic_results_ood, "OOD States (age-shifted)"),
]):
    for i, method in enumerate(methods_to_plot):
        biases = [(results[s][method] - results[s]["True Prevalence"]) * 100
                  for s in scenarios]
        offset = (i - n_methods / 2 + 0.5) * width
        bars = ax.bar(x + offset, biases, width, label=method, color=colors_map[method])
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(scenarios, rotation=45, ha="right")
    if panel_idx == 0:
        ax.set_ylabel("Bias (percentage points)")

ax1.legend(fontsize=8, loc="lower left")
fig_mpl.suptitle("Prevalence Estimation Bias Under Age Distribution Shift: In-Dist vs OOD", fontsize=13)
fig_mpl.tight_layout()
fig_mpl.savefig("../paper/images/figure2_acs_age_shift.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Summary table: bias across all age-shift scenarios
age_shift_summary = []

for setting, results in [("In-Dist", synthetic_results), ("OOD", synthetic_results_ood)]:
    for scenario, estimates in results.items():
        true_prev = estimates["True Prevalence"]
        row = {
            "Setting": setting,
            "Age Distribution": scenario,
            "True Prevalence": f"{true_prev:.1%}",
        }
        for method in methods_to_plot:
            bias_pp = (estimates[method] - true_prev) * 100
            row[f"{method}"] = f"{bias_pp:+.2f}pp"
        age_shift_summary.append(row)

age_shift_summary_df = pd.DataFrame(age_shift_summary).set_index(["Setting", "Age Distribution"])
print("=== Prevalence Estimation Bias Under Age Distribution Shift ===\n")
age_shift_summary_df